In [ ]:
import os
import pandas as pd
import json
import glob
import pprint
from dotenv import load_dotenv
from tqdm import tqdm
import datetime

tqdm.pandas()

import time
import re

load_dotenv('.env')
# For Google Vertex AI: set GOOGLE_APPLICATION_CREDENTIALS and GCP_PROJECT in your .env file
# GOOGLE_APPLICATION_CREDENTIALS=./secrets/your-credentials-file.json
# GCP_PROJECT=your-gcp-project-id


In [ ]:
%load_ext autoreload
%autoreload 2


Ensure that in the root directory, you also have a .env file with the following keys:
- OPENAI_API_KEY
- ANTHROPIC_API_KEY
- GOOGLE_APPLICATION_CREDENTIALS [this should point to a json credentials file for Google Vertex AI]

Use colab secrets feature if you are using a colab notebook instead of a dotenv file

Run the following at the top of the notebook if you are using a colab notebook:
```python
!pip install google-cloud-aiplatform
!pip install langchain
!pip install langchain-openai
!pip install langchain-anthropic
!pip install langchain-google-vertexai
!pip install python-docx
!pip install openai
!pip install anthropic
!pip install langsmith
!pip install python-dotenv

## Data Processing

Steps:
1. Filter out the SCTs without answers
2. For each SCT, generate a prompt for the LLM

In [ ]:
REASON = True
DEBUG = True
FEW_SHOT = False

BASE_PATH = '.'
INPUT_PATH = BASE_PATH + '/data/sct_missing_questions.csv' # points to the cleaned SCTs without answers; orginally file called sct_cleaned.csv
TEMPLATE_PATH = BASE_PATH + '/data/templates'

reason_folder = 'w_reason' if REASON else 'wo_reason'
few_shot_folder = 'few-shot' if FEW_SHOT else 'zero-shot'

OUTPUT_PATH = f'{BASE_PATH}/data/output/{reason_folder}_{few_shot_folder}/'

In [ ]:
df = pd.read_csv(INPUT_PATH, encoding_errors='ignore')

# drop rows where there are missing values for sct_stem
df = df.dropna(subset=['sct_stem', 'question', 'additional_info'])

# print columns of dataframe
if DEBUG:
    print("Columns: ", df.columns)
    print("Number of Questions: ", len(df))

In [ ]:
def generate_prompt_template(guideline: str, testcase: str, reason: bool, few_shot: bool, examples):

  """
  Generate a prompt template based on the given guideline, testcase and examples.

  Args:
    guideline (str): The guideline for the SCT task.
    testcase (str): The case template for which the SCT task is being generated.
    reason (bool): Whether to include the explanation in the prompt.
    few_shot (bool): Whether to include few-shot examples in the prompt.
    examples (dict): A dictionary of examples with keys as the ratings (str) and values as the example text (str).

  Returns:
    str: The generated prompt template.
  """
#  if not reason:
#    guideline = guideline.replace(' and a brief explanation for your choice', '')

  prompt = guideline
  if few_shot:
    prompt += "## Examples with Response Labels"
    for rating in ['-2', '-1', '0', '+1', '+2']:
      prompt += examples[rating]
  prompt += testcase

  return prompt

In [ ]:
examples = {}
#for rating in ['-2', '-1', '0', '+1', '+2']:
#  with open(f'{TEMPLATE_PATH}/example_{rating}.md', 'r') as f:
#    examples[rating] = f.read()

with open(f'{TEMPLATE_PATH}/updated_guideline_sct_cot_clinical_reasoning.md', 'r', encoding='utf-8') as f:
	guideline = f.read()

with open(f'{TEMPLATE_PATH}/testcase.md', 'r', encoding='utf-8') as f:
	testcase = f.read()

In [ ]:
prompt_template = generate_prompt_template(guideline, testcase, REASON, FEW_SHOT, examples)

In [ ]:
prompt_template

In [ ]:
def parse_prompt(row):
	return prompt_template.replace('{{ scenario }}', row['sct_stem']).replace('{{ hypothesis }}', row['question']).replace('{{ additional information }}', row['additional_info'])

df['info_prompt'] = df.apply(parse_prompt, axis=1)

if DEBUG:
	print("Info Prompts Examples: ")
	print(df['info_prompt'].head(1).values)

## Do Question Answering with Langchain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_vertexai import ChatVertexAI
from langchain.schema import HumanMessage, SystemMessage
from langchain_core.rate_limiters import InMemoryRateLimiter


rate_limiter_gemini = InMemoryRateLimiter(
    requests_per_second=0.08,  # <-- Super slow! We can only make a request once every 10 seconds!!
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)

rate_limiter_openai = InMemoryRateLimiter(
    requests_per_second=1,
    check_every_n_seconds=0.1,
    max_bucket_size=10,
)
rate_limiter_anthropic = InMemoryRateLimiter(
    requests_per_second=0.5,
    check_every_n_seconds=0.1,
    max_bucket_size=10,
)


# Define models to run. Add or swap models as needed.
models = [
    (ChatOpenAI(model="gpt-4.1-2025-04-14", api_key=os.getenv("OPENAI_API_KEY"), rate_limiter=rate_limiter_openai), "OpenAI", "gpt-4.1-2025-04-14"),
    # (ChatAnthropic(model="claude-3-5-sonnet-latest", api_key=os.getenv("ANTHROPIC_API_KEY"), rate_limiter=rate_limiter_anthropic), "Anthropic", "claude-3-5-sonnet-latest"),
    # (ChatVertexAI(model="gemini-1.5-pro-002", rate_limiter=rate_limiter_gemini), "VertexAI", "gemini-1.5-pro-002"),
    # (ChatOpenAI(model="o1-2024-12-17", api_key=os.getenv("OPENAI_API_KEY"), rate_limiter=rate_limiter_openai), "OpenAI", "o1-2024-12-17"),
]


if DEBUG:
    print("Testing Models with simple hello prompts")
    for model, provider, model_name in models:
        print(f"Testing Model: {model_name} from {provider}")
        # Execute simple hello for each model
        messages = [HumanMessage(content="Hello")]
        response = model.invoke(messages).content
        print(f"Response: {response}")


In [ ]:
import random, time
import openai
try:                                    # openai-python 0.x
    import openai.error as _oe
    OE = (
        _oe.APIError,
        _oe.Timeout,
        _oe.APIConnectionError,
        _oe.RateLimitError,
    )
except ImportError:                     # openai-python >= 1.0
    OE = (
        openai.APIError,
        openai.APITimeoutError,
        openai.APIConnectionError,
        openai.RateLimitError,
    )

start_time = time.time()
N_SAMPLES = 15  # number of generations per case

params={
    "temperature": 0.7,
    "top_p": 1.0
}

def with_retry(func, *args, **kwargs):
    """Call *func* with exponential back-off on transient OpenAI errors."""
    backoff = 2
    for attempt in range(1, 6):
        try:
            return func(*args, **kwargs)
        except OE as exc:
            if attempt == 5:
                raise                    # give up
            wait = backoff * (1 + random.uniform(-0.3, 0.3))
            print(f"⟳ {type(exc).__name__} – retry {attempt}/5 in {wait:.1f}s")
            time.sleep(wait)
            backoff *= 2

for model, provider, model_name in models:
    date = datetime.datetime.now()
    date_str = date.strftime('%Y-%m-%d-%H-%M-%S')
    response_path = os.path.join(OUTPUT_PATH, f'responses_{provider}_{model_name}_{date_str}.json')

    if not os.path.exists(os.path.dirname(response_path)):
        os.makedirs(os.path.dirname(response_path))

    def invoke_model(row):
        messages = [HumanMessage(content=row["info_prompt"])]
        responses = {}
        for i in range(1, N_SAMPLES + 1):
            response = with_retry(model.invoke, messages, **params)
            if type(response) != str:
                responses[f"response{i}"] = response.content
        return responses

    print('-'*80)
    print(f"Generating responses with {model_name} from {provider}")
    df['response'] = df.progress_apply(invoke_model, axis=1)

    with open(response_path, 'w') as f:
        df_json = df.to_json(orient='records')
        df_json = json.loads(df_json)
        json.dump(df_json, f, indent=4)

end_time = time.time()
elapsed_time = end_time - start_time
print(f"\n⏱️ Finished! Total execution time: {elapsed_time:.2f} seconds")


## Evaluate Responses

In [ ]:
from sklearn.metrics import log_loss
import numpy as np


def cross_entropy(ans_distribution, llm_response):
	# Convert ans_distribution to probabilities
	ans_distribution = ans_distribution / np.sum(ans_distribution, axis=1, keepdims=True)

	# Number of classes is inferred from the prediction shape
	num_instances = ans_distribution.shape[0]
	
	# Compute cross-entropy loss using scikit-learn
	loss = log_loss(llm_response, ans_distribution, labels=[-2, -1, 0, 1, 2]) / num_instances
	
	return loss


def sct_score(ans_distribution, llm_response):
	# Normalize ans_distribution so that the greates score is always 1
	ans_distribution = ans_distribution / np.max(ans_distribution, axis=1, keepdims=True)

	score = 0
	for i in range(len(llm_response)):
		score += ans_distribution[i, llm_response[i] + 2]
		
	return score / len(llm_response)

def percentage_belong_expert_set(ans_distribution, llm_response):
	score = 0

	for i in range(len(llm_response)):
		expert_val = ans_distribution[i, llm_response[i] + 2]
		if expert_val > 0:
			score += 1

	return score / len(llm_response)

def parse_response(response):
	
	rating = response.split("Rating: ")[1][:5]
	pattern = r'\+2|\+1|0|-1|-2'
	matches = re.findall(pattern, rating)
	label = int(''.join(matches))
	assert label in [-2, -1, 0, 1, 2]
	return label

def safe_parse_response(response):
    try:
        rating_match = re.search(r'rating\s*:\s*([+-]?[012])', response, re.IGNORECASE)
        label = int(rating_match.group(1).replace('+', '')) if rating_match else np.nan
        assert label in [-2, -1, 0, 1, 2] if not pd.isna(label) else True
        return label
    except Exception:
        return np.nan


In [ ]:
DEBUG = True
response_files = glob.glob(f'./data/output/sct_w_reason_zero-shot/responses_*.json')

response_files.sort(key=os.path.getmtime, reverse=True)
test_response_file = response_files[0]  # grabs most recent file

data_string = json.load(open(test_response_file, 'r'))
df = pd.DataFrame(data_string)

if DEBUG:
    print("Columns:", df.columns)
    print("Numbr of Questions:", len(df))

    
    # are there any rows where all expert columns are 0
    expert_cols = ['-2', '-1', '0', '1', '2']
    expert_zero_counts = df[expert_cols].sum(axis=1)
    print("Number of expert responses where all expert columns are 0:", expert_zero_counts[expert_zero_counts == 0].count())
    
    student_response_cols = ['Student Raw Count -2', 
                             'Student Raw Count -1', 
                             'Student Raw Count 0', 
                             'Student Raw Count 1', 
                             'Student Raw Count 2',
                             'Student Average Response']
    
    for col in student_response_cols:
        print("Missing Values:", col, df[col].isna().sum())

    print(df['Student Average Response'].value_counts())


In [ ]:
DEBUG = True

response_files = glob.glob(f'./data/output/sct_w_reason_zero-shot/responses_*.json')
eval_data = []

for file in response_files:

    data_dict = {"file_path": file}

    print("-"*80)
    print(f"Evaluating SCT responses from {file}")
    data_string = json.load(open(file, 'r'))
    df = pd.DataFrame(data_string)

    df['response_rating'] = df['response'].apply(safe_parse_response)

    df = df.dropna(subset=['response_rating'])

    print(f"Missing Values for {df['response_rating'].isna().sum()} out of {len(df)}")

    ans_dist = df[['-2', '-1', '0', '1', '2']].to_numpy()
    llm_pred = df['response_rating'].to_numpy().astype(int)

    if DEBUG:
        print("Answer Distribution Shape:", ans_dist.shape)
        print("LLM Prediction Shape:", llm_pred.shape)
        print("Value Counts:", np.unique(llm_pred, return_counts=True))

    data_dict['sct_score'] = sct_score(ans_dist, llm_pred)
    data_dict['perc_belong_expert_set'] = percentage_belong_expert_set(ans_dist, llm_pred)

    print(f"SCT Score: {sct_score(ans_dist, llm_pred)}")
    print(f"% Belonging to Expert Set: {percentage_belong_expert_set(ans_dist, llm_pred)}")

    data_dict['sources_data'] = {}

    sources = df['source'].unique()

    for source in sources:
        print("-"*10)
        print("Evaluating Metrics for Source:", source)
        source_df = df[df['source'] == source]

        source_ans_dist = source_df[['-2', '-1', '0', '1', '2']].to_numpy()
        source_llm_pred = source_df['response_rating'].to_numpy().astype(int)

        data_dict['sources_data'][source] = {}

        data_dict['sources_data'][source]['sct_score'] = sct_score(source_ans_dist, source_llm_pred)
        data_dict['sources_data'][source]['perc_belong_expert_set'] = percentage_belong_expert_set(source_ans_dist, source_llm_pred)

        print("Number of Questions for Source:", len(source_df))
        print(f"SCT Score: {sct_score(source_ans_dist, source_llm_pred)} ")
        print(f"% Belonging to Expert Set: {percentage_belong_expert_set(source_ans_dist, source_llm_pred)}")

    eval_data.append(data_dict)

# write out model data to json file
with open(f'./data/output/sct_w_reason_zero-shot/evaluation_data.json', 'w') as f:
    json.dump(eval_data, f, indent=4)
